In [1]:
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
np.random.seed(42)

num_rows = 5000

cities = ["Abu Dhabi", "Dubai", "Sharjah", "Ajman"]

large_df = pd.DataFrame({
    "total_orders": np.random.randint(1, 30, size=num_rows),
    "total_spent": np.random.uniform(50, 5000, size=num_rows).round(2),
    "successful_payments": np.random.randint(0, 30, size=num_rows),
    "city": np.random.choice(cities, size=num_rows)
})

large_df["average_order_amount"] = (
    large_df["total_spent"] / large_df["total_orders"]
).round(2)

large_df["will_reorder"] = (
    (large_df["total_orders"] >= 5) &
    (large_df["successful_payments"] >= 4) &
    (large_df["total_spent"] >= 500)
).astype(int)

large_df.head()

,total_orders,total_spent,successful_payments,city,average_order_amount,will_reorder
0,7,4739.61,5,Sharjah,677.09,1
1,20,1719.83,0,Sharjah,85.99,0
2,29,3638.66,9,Ajman,125.47,1
3,15,2717.12,28,Abu Dhabi,181.14,1
4,11,350.11,13,Abu Dhabi,31.83,0


In [3]:
large_features_df = pd.get_dummies(
    large_df,
    columns=["city"],
    prefix="cat__city"
)

large_features_df.head()

,total_orders,total_spent,successful_payments,average_order_amount,will_reorder,cat__city_Abu Dhabi,cat__city_Ajman,cat__city_Dubai,cat__city_Sharjah
0,7,4739.61,5,677.09,1,False,False,False,True
1,20,1719.83,0,85.99,0,False,False,False,True
2,29,3638.66,9,125.47,1,False,True,False,False
3,15,2717.12,28,181.14,1,True,False,False,False
4,11,350.11,13,31.83,0,True,False,False,False


In [4]:
large_features_df = large_features_df.rename(columns={
    "total_orders": "num__total_orders",
    "total_spent": "num__total_spent",
    "average_order_amount": "num__average_order_amount",
    "successful_payments": "num__successful_payments"
})

large_features_df.head()

,num__total_orders,num__total_spent,num__successful_payments,num__average_order_amount,will_reorder,cat__city_Abu Dhabi,cat__city_Ajman,cat__city_Dubai,cat__city_Sharjah
0,7,4739.61,5,677.09,1,False,False,False,True
1,20,1719.83,0,85.99,0,False,False,False,True
2,29,3638.66,9,125.47,1,False,True,False,False
3,15,2717.12,28,181.14,1,True,False,False,False
4,11,350.11,13,31.83,0,True,False,False,False


In [5]:
large_features_df.to_csv("../data/processed/features_large.csv", index=False)

print("Large dataset saved successfully.")
print("Shape:", large_features_df.shape)

Large dataset saved successfully.
Shape: (5000, 9)


In [6]:
target_column = "will_reorder"

X_large = large_features_df.drop(columns=[target_column])
y_large = large_features_df[target_column]

print("X_large shape:", X_large.shape)
print("y_large shape:", y_large.shape)
print(y_large.value_counts())

X_large shape: (5000, 8)
y_large shape: (5000,)
will_reorder
1    3407
0    1593
Name: count, dtype: int64


In [7]:
X_large_train, X_large_test, y_large_train, y_large_test = train_test_split(
    X_large,
    y_large,
    test_size=0.2,
    random_state=42,
    stratify=y_large
)

print("X_large_train shape:", X_large_train.shape)
print("X_large_test shape:", X_large_test.shape)
print("y_large_train shape:", y_large_train.shape)
print("y_large_test shape:", y_large_test.shape)

X_large_train shape: (4000, 8)
X_large_test shape: (1000, 8)
y_large_train shape: (4000,)
y_large_test shape: (1000,)


In [8]:
large_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

In [9]:
large_results = []

for model_name, model in large_models.items():
    start_time = time.time()
    
    model.fit(X_large_train, y_large_train)
    y_large_pred = model.predict(X_large_test)
    
    accuracy = accuracy_score(y_large_test, y_large_pred)
    execution_time = time.time() - start_time
    
    large_results.append({
        "model": model_name,
        "accuracy": accuracy,
        "execution_time_seconds": execution_time
    })

large_results_df = pd.DataFrame(large_results)
large_results_df

,model,accuracy,execution_time_seconds
0,Logistic Regression,0.834,0.271442
1,Decision Tree,1.000,0.023774
2,Random Forest,1.000,0.587756


In [10]:
large_cv_results = []

for model_name, model in large_models.items():
    start_time = time.time()
    
    scores = cross_val_score(
        model,
        X_large,
        y_large,
        cv=5,
        scoring="accuracy"
    )
    
    execution_time = time.time() - start_time
    
    large_cv_results.append({
        "model": model_name,
        "cv_mean_accuracy": scores.mean(),
        "cv_std_accuracy": scores.std(),
        "execution_time_seconds": execution_time
    })

large_cv_results_df = pd.DataFrame(large_cv_results)
large_cv_results_df

,model,cv_mean_accuracy,cv_std_accuracy,execution_time_seconds
0,Logistic Regression,0.8358,0.0136,1.392794
1,Decision Tree,1.0000,0.0000,0.072106
2,Random Forest,1.0000,0.0000,2.555100


In [11]:
large_param_grid = {
    "max_depth": [2, 3, 4, 5, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

large_grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=large_param_grid,
    cv=5,
    scoring="accuracy"
)

start_time = time.time()

large_grid_search.fit(X_large_train, y_large_train)

large_grid_search_time = time.time() - start_time

print("Best parameters:", large_grid_search.best_params_)
print("Best cross-validation accuracy:", large_grid_search.best_score_)
print("GridSearch execution time:", large_grid_search_time)

Best parameters: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best cross-validation accuracy: 1.0
GridSearch execution time: 2.5356366634368896


In [12]:
best_large_model = large_grid_search.best_estimator_

y_large_pred_best = best_large_model.predict(X_large_test)

best_large_accuracy = accuracy_score(y_large_test, y_large_pred_best)

print("Best large model:", best_large_model)
print("Test accuracy:", best_large_accuracy)
print("\nClassification Report:")
print(classification_report(y_large_test, y_large_pred_best))

Best large model: DecisionTreeClassifier(max_depth=3, random_state=42)
Test accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       319
           1       1.00      1.00      1.00       681

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [13]:
joblib.dump(best_large_model, "best_model_large.pkl")

print("Large-data best model saved as best_model_large.pkl")

Large-data best model saved as best_model_large.pkl


In [14]:
large_results_df.to_csv("large_baseline_model_results.csv", index=False)
large_cv_results_df.to_csv("large_cross_validation_results.csv", index=False)

print("Large experiment results saved successfully.")

Large experiment results saved successfully.
